In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
while not (PROJECT_ROOT / 'setup.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Cannot find project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from solve_dae.integrate import solve_dae

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts" / "Experimental"))

from simulate_HFR import build_cell, build_atol, soc_cutoff_event
from rfbmodel1D.parameters.params import cathodeparams, anodeparams, membraneparams, cellparams

## Settings

In [ ]:
TF     = 1800   # Simulation time horizon [s]
RTOL   = 1e-5
STRIDE = 1      # Sub-sample experimental points (1 = use all)

# Temperature -> colour mapping
T_COLOR = {24.0: "goldenrod", 32.0: "forestgreen", 40.0: "firebrick"}

RES_DIR = ROOT / "results" / "test fitting"
SIM_DIR = RES_DIR / "hfr_sims"       # cached .npz results
FIG_DIR = ROOT / "figures" / "hfr_all"
SIM_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Load test configurations from fit JSONs

In [ ]:
hfr_df = pd.read_csv(ROOT / "data" / "experimental" / "HFR.csv")

# Discover tests that have at least one fit JSON
test_folders = sorted(
    RES_DIR.glob("test *"),
    key=lambda p: int(p.name.split()[-1])
)
ALL_TESTS = [
    int(p.name.split()[-1]) for p in test_folders
    if sorted(p.glob(f"fit_v2_test{int(p.name.split()[-1])}_*.json"))
]

test_configs = {}
for t in ALL_TESTS:
    fit_dir  = RES_DIR / f"test {t}"
    fit_path = sorted(fit_dir.glob(f"fit_v2_test{t}_*.json"))[-1]
    with open(fit_path) as f:
        _hfr = json.load(f)
    _p = _hfr["params"]
    _s = _hfr["settings"]

    hfr_params = dict(
        k_sorp_vw       = _p["k_vw_sorp"],
        k_desorp_vw     = _p["S_desorp"] * _p["k_vw_sorp"],
        k_sorp_lw       = _p["k_lw_sorp"],
        k_desorp_lw     = _p["S_desorp"] * _p["k_lw_sorp"],
        k_ca_lw         = _p["k_ca"],
        sf_D            = _s["sf_D"],
        vanadium_uptake = _s.get("vanadium_uptake", False),
    )
    R_EXTRA = _p["R_extra (Ω·cm²)"] / 1e4   # Ω·cm² -> Ω·m²

    # Optional E-cell fit for cathode/anode kinetics
    ecell_jsons = sorted(fit_dir.glob(f"fit_e_cell_test{t}_*.json"))
    if ecell_jsons:
        with open(ecell_jsons[-1]) as f:
            _ec = json.load(f)
        k_ca_cathode = _ec["params"]["k_ca"]
        j0_an        = _ec["params"]["j0_an"]
    else:
        k_ca_cathode = None
        j0_an        = None

    d   = hfr_df[hfr_df["test_no"] == t]
    soc = float(d["SOC"].iloc[0])
    T   = float(d["T(°C)"].iloc[0])
    Q_v = float(d["Q_v (ml/min)"].iloc[0])

    j_ch  = d[d["mode"] == "charge"]   ["j_appl (A/m2)"].values[::STRIDE]
    j_dis = d[d["mode"] == "discharge"]["j_appl (A/m2)"].values[::STRIDE]
    H_ch  = d[d["mode"] == "charge"]   ["value (ohm cm2)"].values[::STRIDE]
    H_dis = d[d["mode"] == "discharge"]["value (ohm cm2)"].values[::STRIDE]

    test_configs[t] = dict(
        soc=soc, T=T, Q_v=Q_v,
        j_charge=j_ch, j_discharge=j_dis,
        HFR_exp_ch=H_ch, HFR_exp_dis=H_dis,
        hfr=hfr_params, R_EXTRA=R_EXTRA,
        k_ca_cathode=k_ca_cathode, j0_an=j0_an,
    )
    ecell_tag = "  [E-cell loaded]" if ecell_jsons else ""
    print(f"Test {t:2d}: T={T:.0f}°C  SOC={soc:.1f}  "
          f"Q_v={Q_v:.0f} ml/min  ch={len(j_ch)}pts  "
          f"dis={len(j_dis)}pts  R_extra={R_EXTRA*1e4:.4f} Ω·cm²{ecell_tag}")

## Simulation functions

In [ ]:
def _apply_params(cfg: dict) -> None:
    h = cfg["hfr"]
    membraneparams.k_sorp_vw        = h["k_sorp_vw"]
    membraneparams.k_desorp_vw      = h["k_desorp_vw"]
    membraneparams.k_sorp_lw        = h["k_sorp_lw"]
    membraneparams.k_desorp_lw      = h["k_desorp_lw"]
    membraneparams.k_ca_lw          = h["k_ca_lw"]
    membraneparams.sf_D             = h["sf_D"]
    cellparams.vanadium_uptake      = h["vanadium_uptake"]
    if cfg["k_ca_cathode"] is not None:
        cathodeparams.reaction_rate_constant = cfg["k_ca_cathode"]
    if cfg["j0_an"] is not None:
        anodeparams.j_0 = cfg["j0_an"]


def run_sim(j_appl: float, cfg: dict) -> float:
    """Steady-state simulation at j_appl [A/m²]. Returns HFR [Ω·cm²]."""
    _apply_params(cfg)
    cell, y0, yp0 = build_cell(cfg["soc"], j_appl, cfg["T"], cfg["Q_v"])
    atol          = build_atol(cell)
    jac_sparsity  = cell.compute_jac_sparsity(y0, yp0)
    t0 = time.time()
    sol = solve_dae(
        cell.dae_residual_full_cell,
        (0.0, TF), y0, yp0,
        method="BDF", atol=atol, rtol=RTOL,
        events=soc_cutoff_event, jac_sparsity=jac_sparsity,
    )
    if sol.t[-1] < 0.99 * TF:
        print(f"    [warn] t_final={sol.t[-1]:.1f} s (event triggered)")

    kappa_io_ca = cell.cathode.kappa_io
    sigma_ca    = np.ones_like(kappa_io_ca) * cell.cathode.eff_conductivity
    dx_ca       = cell.mesh.cathode_mesh.dx * cell.cathode.thickness
    R_cathode   = sum(dx_ca / (sigma_ca + kappa_io_ca))

    kappa_io_m  = cell.membrane.kappa_io
    dx_m        = cell.mesh.membrane_mesh.dx * cell.membrane.thickness
    R_membrane  = sum(dx_m / kappa_io_m)

    R_ACL  = cell.anode.thickness  / cell.anode.eff_conductivity
    R_AGDL = cell.agdl.thickness   / cell.agdl.eff_conductivity
    R_BP   = cell.thickness_BP / cell.sigma_BP
    R_CC   = cell.thickness_CC / cell.sigma_CC
    HFR    = (R_cathode + R_membrane + R_ACL + R_AGDL + R_BP + R_CC + cfg["R_EXTRA"]) * 1e4

    print(f"  j={j_appl:+8.0f} A/m²  HFR={HFR:.4f} Ω·cm²  ({time.time()-t0:.1f}s)")
    return HFR


def interior_points(arr, n=1):
    """Insert n equally-spaced points between each consecutive pair."""
    pts = []
    for a, b in zip(arr[:-1], arr[1:]):
        pts.extend(np.linspace(a, b, n + 2)[1:-1])
    return np.array(pts)

## Run simulations
Results are cached as `.npz` in `results/test fitting/hfr_sims/`.  
Re-run a test by deleting its `.npz` file.

In [ ]:
sim_results = {}

for t in ALL_TESTS:
    npz_path = SIM_DIR / f"hfr_test{t}.npz"
    cfg      = test_configs[t]

    if npz_path.exists():
        print(f"Test {t:2d}: loading cache from {npz_path.name}")
        raw = dict(np.load(npz_path))
        sim_results[t] = {k: (float(v) if v.ndim == 0 else v) for k, v in raw.items()}
        continue

    print(f"\n{'='*60}")
    print(f"Test {t}  |  T={cfg['T']}°C  SOC={cfg['soc']}  Q_v={cfg['Q_v']} ml/min")
    print(f"{'='*60}")

    # -- Charge ---------------------------------------------------------------
    print("-- Charge --")
    HFR_ch = [run_sim(float(j), cfg) for j in cfg["j_charge"]]

    j_ch_fill   = interior_points(cfg["j_charge"], n=1)
    print("-- Charge fill --")
    HFR_ch_fill = [run_sim(float(j), cfg) for j in j_ch_fill]

    j_model_ch   = np.concatenate([cfg["j_charge"], j_ch_fill])
    HFR_model_ch = np.concatenate([HFR_ch, HFR_ch_fill])
    idx = np.argsort(j_model_ch)
    j_model_ch, HFR_model_ch = j_model_ch[idx], HFR_model_ch[idx]

    # -- Discharge -------------------------------------------------------------
    print("-- Discharge --")
    HFR_dis = [run_sim(-float(j), cfg) for j in cfg["j_discharge"]]

    j_dis_fill   = interior_points(cfg["j_discharge"], n=3)
    print("-- Discharge fill --")
    HFR_dis_fill = [run_sim(-float(j), cfg) for j in j_dis_fill]

    j_model_dis   = np.concatenate([cfg["j_discharge"], j_dis_fill])
    HFR_model_dis = np.concatenate([HFR_dis, HFR_dis_fill])
    idx = np.argsort(j_model_dis)
    j_model_dis, HFR_model_dis = j_model_dis[idx], HFR_model_dis[idx]

    # -- Metrics ---------------------------------------------------------------
    HFR_exp_all = np.concatenate([cfg["HFR_exp_ch"], cfg["HFR_exp_dis"]])
    HFR_sim_all = np.concatenate([HFR_ch, HFR_dis])
    rmse = float(np.sqrt(np.mean((HFR_sim_all - HFR_exp_all) ** 2)))
    mape = float(np.mean(np.abs((HFR_sim_all - HFR_exp_all) / HFR_exp_all)) * 100)
    print(f"  RMSE = {rmse:.4f} Ω·cm²   MAPE = {mape:.2f}%")

    sim_results[t] = dict(
        j_model_ch   = j_model_ch,
        HFR_model_ch = HFR_model_ch,
        j_model_dis  = j_model_dis,
        HFR_model_dis= HFR_model_dis,
        HFR_sim_ch   = np.array(HFR_ch),
        HFR_sim_dis  = np.array(HFR_dis),
        rmse         = rmse,
        mape         = mape,
    )

    np.savez_compressed(npz_path, **sim_results[t])
    print(f"  Saved -> {npz_path}")

## Metrics summary

In [ ]:
print(f"{'Test':>5}  {'T(°C)':>6}  {'SOC':>5}  {'Q_v':>6}  {'RMSE (Ω·cm²)':>14}  {'MAPE (%)':>10}")
print("-" * 58)
for t in ALL_TESTS:
    cfg  = test_configs[t]
    r    = sim_results[t]
    print(f"{t:>5}  {cfg['T']:>6.0f}  {cfg['soc']:>5.1f}  "
          f"{cfg['Q_v']:>6.0f}  {r['rmse']:>14.4f}  {r['mape']:>10.2f}")

## Plots
**Colour code:** yellow (goldenrod) = 24 °C · green (forestgreen) = 32 °C · red (firebrick) = 40 °C  
Legend shows experimental markers only; model is the solid line (colour inferred).

In [ ]:
plt.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size':        9,
    'axes.labelsize':   9,
    'axes.titlesize':   8,
    'xtick.labelsize':  8,
    'ytick.labelsize':  8,
    'legend.fontsize':  8,
    'axes.linewidth':   0.6,
    'xtick.direction':  'in',
    'ytick.direction':  'in',
    'xtick.top':        True,
    'ytick.right':      True,
    'lines.linewidth':  1.2,
    'savefig.dpi':      600,
    'pdf.fonttype':     42,
    'ps.fonttype':      42,
})

# Legend proxy handles – grey neutral markers, colour already on the line
_LEG = [
    plt.Line2D([0], [0], marker='o', color='none',
               mfc='#555555', mec='#555555', ms=5, mew=0.9,
               ls='none', label='Exp., charge'),
    plt.Line2D([0], [0], marker='o', color='none',
               mfc='none',    mec='#555555', ms=5, mew=0.9,
               ls='none', label='Exp., discharge'),
]


def plot_group(tests, label_overrides=None, save_name=None, width_per=2.36):
    """
    tests          : list of test numbers to plot side by side
    label_overrides: {test_no: str} appended in parentheses below the title
    save_name      : PDF stem saved to figures/hfr_all/
    width_per      : inches per subplot
    """
    n    = len(tests)
    fig, axes = plt.subplots(1, n, figsize=(n * width_per, 3.0), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, t in zip(axes, tests):
        cfg   = test_configs[t]
        r     = sim_results[t]
        color = T_COLOR[cfg["T"]]

        # Experimental data – filled = charge, open = discharge
        ax.plot(cfg["j_discharge"], cfg["HFR_exp_dis"],
                'o', ms=5, mfc='none', mec=color, mew=0.9, zorder=3)
        ax.plot(cfg["j_charge"], cfg["HFR_exp_ch"],
                'o', ms=5, mfc=color,  mec=color, mew=0.9, zorder=3)

        # Model lines (no legend entry)
        ax.plot(r["j_model_ch"],  r["HFR_model_ch"],  '-', color=color, lw=1.3)
        ax.plot(r["j_model_dis"], r["HFR_model_dis"], '-', color=color, lw=1.3)

        extra = f"\n({label_overrides[t]})" if (label_overrides and t in label_overrides) else ""
        ax.set_title(
            f"Test {t}{extra}\n"
            f"SOC={cfg['soc']:.0%}  {cfg['T']:.0f}°C  {cfg['Q_v']:.0f} ml/min\n"
            f"RMSE={r['rmse']:.3f} Ω·cm²  MAPE={r['mape']:.1f}%",
            pad=3,
        )
        ax.set_xlabel(r'$j$ / A m$^{-2}$')
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

    axes[0].set_ylabel(r'HFR / $\Omega$ cm$^2$')
    axes[-1].legend(handles=_LEG, frameon=False, loc='upper right')

    fig.tight_layout(pad=0.5)

    if save_name:
        out = FIG_DIR / f"{save_name}.pdf"
        fig.savefig(out, bbox_inches='tight')
        print(f"Saved -> {out}")

    plt.show()

### All groups — grouped by SOC and Q_v (only T varies within each panel)

| Group | Tests | SOC | Q_v (ml/min) | T (°C) |
|-------|-------|-----|--------------|--------|
| 1 | 1, 2, 3 | 0.1 | 20 | 24, 32, 40 |
| 2 | 4, 5, 6 | 0.1 | 45 | 24, 32, 40 |
| 3 | 7, 8, 9 | 0.1 | 70 | 24, 32, 40 |
| 4 | 10, 11, 12 | 0.5 | 20 | 24, 32, 40 |
| 5 | 13, 14, 15, 16, 17 | 0.5 | 45 | 24, 32*, 32*, 32, 40 — 5-panel |
| 6 | 18, 19, 20 | 0.5 | 70 | 24, 32, 40 |
| 7 | 21, 22, 23 | 0.9 | 20 | 24, 32, 40 |
| 8 | 24, 25, 26 | 0.9 | 45 | 24, 32, 40 |
| 9 | 27, 28, 29 | 0.9 | 70 | 24, 32, 40 |

\* Tests 14 and 15 are repeats at 32 °C.

In [ ]:
# Groups defined by (SOC, Q_v) — within each group only T varies (24->32->40°C).
# Note: test 24 conditions are SOC=0.9, Q_v=45, T=24°C (CSV may show different values).
PLOT_GROUPS = [
    # SOC=0.1
    dict(tests=[1,  2,  3],          labels=None,                             name="HFR_SOC10_Qv20"),
    dict(tests=[4,  5,  6],          labels=None,                             name="HFR_SOC10_Qv45"),
    dict(tests=[7,  8,  9],          labels=None,                             name="HFR_SOC10_Qv70"),
    # SOC=0.5
    dict(tests=[10, 11, 12],         labels=None,                             name="HFR_SOC50_Qv20"),
    dict(tests=[13, 14, 15, 16, 17], labels={14: "Repeat 1", 15: "Repeat 2"}, name="HFR_SOC50_Qv45", width_per=2.0),
    dict(tests=[18, 19, 20],         labels=None,                             name="HFR_SOC50_Qv70"),
    # SOC=0.9
    dict(tests=[21, 22, 23],         labels=None,                             name="HFR_SOC90_Qv20"),
    dict(tests=[24, 25, 26],         labels=None,                             name="HFR_SOC90_Qv45"),
    dict(tests=[27, 28, 29],         labels=None,                             name="HFR_SOC90_Qv70"),
]

for grp in PLOT_GROUPS:
    tests = [t for t in grp["tests"] if t in sim_results]
    plot_group(
        tests,
        label_overrides=grp.get("labels"),
        save_name=grp["name"],
        width_per=grp.get("width_per", 2.36),
    )